# Lab 4.2: KV Cache Compression[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.2_kv_cache_compression/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open_in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.2_kv_cache_compression/lab.ipynb)Measure real KV cache memory at different quantization levels and visualize the compression/quality tradeoff. We implement TurboQuant's rotation trick, compare against naive quantization, and show exactly where compression helps vs hurts.

In [ ]:
# --- Setup ---import torchimport numpy as npimport matplotlib.pyplot as plt# Device detection: use GPU if availabledevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")# Model config: Mistral-7B / Llama-3.1-8B architectureN_LAYERS = 32       # number of transformer layersN_KV_HEADS = 8      # GQA: 8 KV heads (not 32 query heads)HEAD_DIM = 128      # dimension per headDTYPE_BYTES = 2     # FP16 = 2 bytes per element

## Experiment 1: KV Cache Memory at Different Bit-WidthsCompute exact memory for a Mistral-7B-class model across sequence lengths and quantization levels.

In [ ]:
# KV cache memory formula:# memory_bytes = 2 (K+V) * n_layers * n_kv_heads * seq_len * head_dim * (bits/8)def kv_memory_gb(seq_len, bits=16, n_layers=N_LAYERS, n_kv_heads=N_KV_HEADS, head_dim=HEAD_DIM):    """Compute KV cache memory in GB for given config."""    # 2 for K and V tensors    return 2 * n_layers * n_kv_heads * seq_len * head_dim * (bits / 8) / (1024**3)# Sweep sequence lengths and bit-widthsseq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]bit_widths = [16, 8, 4, 3]colors = {'16': '#6b7280', '8': '#2563eb', '4': '#f59e0b', '3': '#dc2626'}fig, ax = plt.subplots(figsize=(10, 5))for bits in bit_widths:    # Compute memory for each sequence length    mem = [kv_memory_gb(s, bits) for s in seq_lengths]    ax.plot(seq_lengths, mem, 'o-', label=f'{bits}-bit ({16/bits:.1f}x compression)',            color=colors[str(bits)], linewidth=2)# Mark typical GPU memory limitsax.axhline(y=24, color='red', linestyle='--', alpha=0.5, label='A10G VRAM (24 GB)')ax.axhline(y=80, color='purple', linestyle='--', alpha=0.5, label='H100 VRAM (80 GB)')ax.set_xlabel('Sequence Length (tokens)')ax.set_ylabel('KV Cache Memory (GB)')ax.set_title('KV Cache Memory vs Sequence Length (Mistral-7B, batch=1)')ax.legend()ax.set_xscale('log', base=2)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()# Print exact values at key pointsprint(f"{'Seq Len':>8} | {'FP16':>8} | {'8-bit':>8} | {'4-bit':>8} | {'3-bit':>8}")print('-' * 50)for s in [2048, 8192, 32768]:    row = f"{s:>8} | "    row += ' | '.join(f"{kv_memory_gb(s, b)*1000:.0f} MB" for b in bit_widths)    print(row)

## Experiment 2: TurboQuant Rotation vs Naive QuantizationThe key insight: random rotation eliminates outlier dimensions, making scalar quantization near-optimal. We measure reconstruction error with and without rotation.

In [ ]:
def random_rotation_matrix(d, device):    """Generate random orthogonal matrix via QR decomposition."""    H = torch.randn(d, d, device=device)    Q, _ = torch.linalg.qr(H)    return Qdef scalar_quantize(x, bits):    """Symmetric per-row scalar quantization."""    qmax = (1 << (bits - 1)) - 1    scale = x.abs().amax(dim=-1, keepdim=True) / qmax    x_q = (x / scale.clamp(min=1e-8)).round().clamp(-qmax, qmax)    return x_q, scaledef dequantize(x_q, scale):    """Reconstruct from quantized values."""    return x_q * scale# Simulate a realistic KV cache slice with outlier dimensions# (common in trained models: embedding dims have non-uniform magnitudes)seq_len = 4096kv = torch.randn(seq_len, HEAD_DIM, device=device)kv[:, 0] *= 8   # outlier dimension 0kv[:, 1] *= 6   # outlier dimension 1kv[:, 2] *= 4   # moderate outlier# Generate rotation matrix (computed once, reused for all tokens)R = random_rotation_matrix(HEAD_DIM, device)# Compare reconstruction error across bit-widthsresults = []for bits in [2, 3, 4, 8]:    # Naive: quantize directly    nq, ns = scalar_quantize(kv, bits)    naive_recon = dequantize(nq, ns)    naive_mse = (kv - naive_recon).pow(2).mean().item()    # TurboQuant: rotate then quantize    rotated = kv @ R    tq, ts = scalar_quantize(rotated, bits)    turbo_recon = dequantize(tq, ts) @ R.T  # inverse rotation    turbo_mse = (kv - turbo_recon).pow(2).mean().item()    # Cosine similarity (measures direction preservation for attention)    naive_cos = torch.nn.functional.cosine_similarity(        kv.flatten().unsqueeze(0), naive_recon.flatten().unsqueeze(0)).item()    turbo_cos = torch.nn.functional.cosine_similarity(        kv.flatten().unsqueeze(0), turbo_recon.flatten().unsqueeze(0)).item()    results.append((bits, naive_mse, turbo_mse, naive_cos, turbo_cos))    print(f"{bits}-bit | Naive MSE: {naive_mse:.5f}, cos={naive_cos:.6f} | "          f"TurboQuant MSE: {turbo_mse:.5f}, cos={turbo_cos:.6f} | "          f"Improvement: {naive_mse/turbo_mse:.1f}x")

## Experiment 3: Per-Dimension Error DistributionVisualize why rotation matters: naive quantization concentrates error on outlier dimensions, while rotation spreads it uniformly.

In [ ]:
# Compute per-dimension MSE for 4-bit quantizationnq4, ns4 = scalar_quantize(kv, 4)naive4_recon = dequantize(nq4, ns4)naive_dim_mse = (kv - naive4_recon).pow(2).mean(dim=0).cpu().numpy()rotated4 = kv @ Rtq4, ts4 = scalar_quantize(rotated4, 4)turbo4_recon = dequantize(tq4, ts4) @ R.Tturbo_dim_mse = (kv - turbo4_recon).pow(2).mean(dim=0).cpu().numpy()fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)# Naive: outlier dims have much higher errorax1.bar(range(HEAD_DIM), naive_dim_mse, color='#dc2626', alpha=0.7, width=1.0)ax1.set_ylabel('MSE per dimension')ax1.set_title('Naive 4-bit: error concentrated on outlier dimensions (dims 0,1,2)')ax1.set_ylim(0, max(naive_dim_mse) * 1.2)# TurboQuant: error is uniform across all dimensionsax2.bar(range(HEAD_DIM), turbo_dim_mse, color='#2563eb', alpha=0.7, width=1.0)ax2.set_xlabel('Dimension index')ax2.set_ylabel('MSE per dimension')ax2.set_title('TurboQuant 4-bit: rotation spreads error uniformly (no spikes)')ax2.set_ylim(0, max(naive_dim_mse) * 1.2)plt.tight_layout()plt.show()# Quantify the differenceprint(f"Naive dim-error std:      {naive_dim_mse.std():.6f} (high = non-uniform = bad)")print(f"TurboQuant dim-error std: {turbo_dim_mse.std():.6f} (low = uniform = good)")print(f"Uniformity improvement:   {naive_dim_mse.std()/turbo_dim_mse.std():.1f}x")

## Experiment 4: Compression-Quality Tradeoff CurveThe key chart: plot memory savings against reconstruction quality to find the sweet spot for your workload.

In [ ]:
# Sweep bit-widths from 2 to 16 and measure quality metricsbit_range = [2, 3, 4, 5, 6, 8, 16]compression_ratios = []cosine_sims_naive = []cosine_sims_turbo = []sqnr_naive = []  # signal-to-quantization-noise ratiosqnr_turbo = []signal_power = kv.pow(2).mean().item()for bits in bit_range:    compression_ratios.append(16.0 / bits)    if bits == 16:        cosine_sims_naive.append(1.0)        cosine_sims_turbo.append(1.0)        sqnr_naive.append(float('inf'))        sqnr_turbo.append(float('inf'))        continue    # Naive    nq, ns = scalar_quantize(kv, bits)    nr = dequantize(nq, ns)    n_mse = (kv - nr).pow(2).mean().item()    cosine_sims_naive.append(torch.nn.functional.cosine_similarity(        kv.flatten().unsqueeze(0), nr.flatten().unsqueeze(0)).item())    sqnr_naive.append(10 * np.log10(signal_power / n_mse))    # TurboQuant    rot = kv @ R    tq, ts = scalar_quantize(rot, bits)    tr = dequantize(tq, ts) @ R.T    t_mse = (kv - tr).pow(2).mean().item()    cosine_sims_turbo.append(torch.nn.functional.cosine_similarity(        kv.flatten().unsqueeze(0), tr.flatten().unsqueeze(0)).item())    sqnr_turbo.append(10 * np.log10(signal_power / t_mse))fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))# Left: cosine similarity vs compression ratioax1.plot(compression_ratios, cosine_sims_naive, 'o--', color='#dc2626',         label='Naive quantization', linewidth=2)ax1.plot(compression_ratios, cosine_sims_turbo, 's-', color='#2563eb',         label='TurboQuant (with rotation)', linewidth=2)ax1.axhline(y=0.999, color='green', linestyle=':', alpha=0.7, label='0.999 threshold')ax1.set_xlabel('Compression Ratio (x)')ax1.set_ylabel('Cosine Similarity')ax1.set_title('Quality vs Compression: Higher is Better')ax1.legend()ax1.grid(True, alpha=0.3)# Right: SQNR vs compression ratiosqnr_n_plot = [s if s != float('inf') else 80 for s in sqnr_naive]sqnr_t_plot = [s if s != float('inf') else 80 for s in sqnr_turbo]ax2.plot(compression_ratios, sqnr_n_plot, 'o--', color='#dc2626',         label='Naive', linewidth=2)ax2.plot(compression_ratios, sqnr_t_plot, 's-', color='#2563eb',         label='TurboQuant', linewidth=2)ax2.axhline(y=30, color='green', linestyle=':', alpha=0.7, label='30 dB (good)')ax2.set_xlabel('Compression Ratio (x)')ax2.set_ylabel('SQNR (dB)')ax2.set_title('Signal-to-Noise: Higher is Better')ax2.legend()ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()# Print summary tableprint(f"{'Bits':>4} | {'Ratio':>5} | {'Naive cos':>10} | {'TQ cos':>10} | {'Naive SQNR':>11} | {'TQ SQNR':>9}")print('-' * 60)for i, bits in enumerate(bit_range):    sn = f"{sqnr_naive[i]:.1f}" if sqnr_naive[i] != float('inf') else "inf"    st = f"{sqnr_turbo[i]:.1f}" if sqnr_turbo[i] != float('inf') else "inf"    print(f"{bits:>4} | {compression_ratios[i]:>5.1f} | {cosine_sims_naive[i]:>10.6f} | "          f"{cosine_sims_turbo[i]:>10.6f} | {sn:>11} | {st:>9}")

## Experiment 5: Batch Scaling -- How Many More Sequences Fit?The practical payoff: quantizing the KV cache lets you serve more concurrent users on the same GPU.

In [ ]:
# Calculate max batch size at different quantization levels# GPU memory budget after weights are loadedgpu_memory_gb = 24.0  # A10Gmodel_weights_gb = 14.0  # Mistral-7B in FP16available_for_kv = gpu_memory_gb - model_weights_gb  # ~10 GB for KV cacheseq_len_serve = 4096  # typical serving context lengthmax_batches = {}for bits in [16, 8, 4, 3]:    # Memory per sequence = KV cache for one request    per_seq_gb = kv_memory_gb(seq_len_serve, bits)    max_batch = int(available_for_kv / per_seq_gb)    max_batches[bits] = max_batch    print(f"{bits:>2}-bit: {per_seq_gb*1000:.0f} MB/seq -> max batch = {max_batch} "          f"({max_batch / max_batches[16]:.1f}x vs FP16)")# Visualizefig, ax = plt.subplots(figsize=(8, 5))bars = ax.bar([f'{b}-bit' for b in max_batches.keys()],              max_batches.values(),              color=['#6b7280', '#2563eb', '#f59e0b', '#dc2626'])# Add value labels on barsfor bar, val in zip(bars, max_batches.values()):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,            str(val), ha='center', fontweight='bold')ax.set_xlabel('KV Cache Quantization Level')ax.set_ylabel('Max Concurrent Sequences')ax.set_title(f'Batch Capacity on A10G (24GB) | Mistral-7B | seq_len={seq_len_serve}')ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()

## Key Takeaways1. **TurboQuant rotation eliminates outlier dimensions**, making 3-4 bit quantization near-lossless (cosine sim > 0.999)2. **Memory scales linearly with bit-width**: 3-bit gives 5.3x compression vs FP16, directly translating to 5x more concurrent sequences3. **The sweet spot is 3-4 bits with rotation**: below 3 bits quality degrades rapidly, above 4 bits you're leaving memory on the table4. **Outlier dimensions are the enemy**: without rotation, a single outlier dim can dominate the quantization scale and waste precision on all other dims5. **Practical impact**: on an A10G serving Mistral-7B at 4K context, 3-bit KV cache enables ~5x more concurrent requests than FP16